# Storm event in detail

One storm with everything we have, to see where the nearshore differences come
from:

1. **Forcing**: offshore waves at the goals, wind and water level (as given to
   the dynamic run).
2. **Buoys**: records against the offshore hindcast (closest goal), the
   metamodel without and with the wind correction, and the dynamic run (wind and
   water level): metrics, storm peaks, time series.
3. **Wind correction**: at a few buoys, the wind sea the correction sees, the
   wind, how developed the sea is, the wind–wave angle and `K`, against the
   training range (05b).
4. **Maps at the peak**: the dynamic run at every site of a map set, the
   metamodel without / with the correction minus it, and `K`.

The metamodel is read from the reconstruction cache (missing months are
computed, whole months). §3 recomputes the event without the cache
(`mm.wind_sea_diagnostics`). `K` is a tree model, so tiny batch differences can
move it: compared with the month-long cache it can differ a little.

**Prerequisites**: fitted models (06), wind correction (05b), the offshore
source (07a/b), the dynamic run of the event's year (08), processed buoys (03).

In [ ]:
import os
import sys

# GDAL/PROJ data paths when the kernel was not started through `conda activate`
os.environ.setdefault("GDAL_DATA", os.path.join(sys.prefix, "share", "gdal"))
os.environ.setdefault("PROJ_DATA", os.path.join(sys.prefix, "share", "proj"))

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from bluemath_tk.waves.hywaves.plots import plot_event_maps, plot_wind_sea_event
from utils import metamodel as mm
from utils import simulation as sim
from utils import validation as val
from utils import wind
from utils.config import load_goals, open_offshore_waves, open_water_level, open_wind
from utils.geometry import attach_site_meta

## Configuration

In [ ]:
event = slice("2013-12-04", "2013-12-07")  # a storm inside the dynamic run
offshore_name = "WHACS"  # offshore_waves entry (the dynamic run's forcing)
mda_n = 450  # must match a fitted mda_candidates entry
mm_set = "buoys"  # metamodel set at the buoys
map_set = "isobaths"  # metamodel set of the maps at the peak
show_buoys = None  # None: the 3 buoys with the highest peak

dynamic_path = sim.cases_dir("dynamic") / f"Dynamic_Cases_{offshore_name}_2013.nc"
wind_labels = {False: "Metamodel · wind off", True: "Metamodel · wind on"}
print(f"event   : {event.start} to {event.stop}")
print(f"dynamic : {dynamic_path} (exists: {dynamic_path.is_file()})")

## 1 · Forcing

Offshore waves at the goals, wind and water level, as given to the dynamic run.

In [ ]:
offshore = open_offshore_waves(offshore_name).sel(time=event).load()
setup = sim.HyWavesWrapperSetup.from_case(load_goals())
fig = setup.plot_dynamic_configuration(
    offshore, offshore_name=offshore_name, wind=open_wind(event), water_level=open_water_level()
)
plt.show()

## 2 · Buoys

Buoy records in the event, the metamodel (cache) without and with the wind correction, the offshore hindcast at the closest goal and the dynamic run. Metrics: median over the buoys.

In [ ]:
buoy = val.load_buoy_observations(event)
sites = list(buoy.sites.values)
show_buoys = show_buoys or list(buoy["hs"].max("time").to_series().nlargest(3).index)
print(f"{len(sites)} buoys with data; shown in detail: {show_buoys}")

# Resources for the reconstructions below (at most 75% of this machine)
res = mm.reconstruction_resources(
    event, mda_n, metamodel=mm_set, spectral=True, offshore_name=offshore_name,
    wl="forcing", save_spectra=False, wind_correction=True,
)

In [ ]:
metamodel = {
    label: mm.reconstruct(
        event, mda_n, metamodel=mm_set, spectral=True, offshore_name=offshore_name,
        wl="forcing", save_spectra=False, wind_correction=on, resources=res,
    )
    for on, label in wind_labels.items()
}
dynamic = attach_site_meta(xr.open_dataset(dynamic_path).sel(time=event).load())
offshore_hindcast, _ = val.load_offshore_hindcast_at_buoys(event, sites, offshore_names=[offshore_name])
estimates = {**offshore_hindcast, **metamodel, val.DYNAMIC_REFERENCE_LABEL: dynamic.sel(sites=sites)}

results = val.compare_all(buoy, estimates, variables=["hs", "tp"], reference_label="Buoy")
val.metrics_summary(results).groupby(["label", "model"])[["n", "bias", "rmse", "r2"]].median().round(3)

Storm peak at each buoy: the largest Hs of each source in the event, its
difference to the buoy peak (m) and its lag (h, positive = later than the buoy).

In [ ]:
peaks = val.event_peaks(buoy, estimates)
diffs = peaks[[c for c in peaks if c.endswith(" - Buoy") or c.endswith(" lag (h)")]]
display(diffs.median().round(2).to_frame("median over the buoys").T)
peaks[["Buoy", *[c for c in peaks if c.endswith(" - Buoy")]]].round(2)

In [ ]:
for site in show_buoys:
    val.plot_buoy_timeseries(buoy, estimates, site, variables=["hs", "tp", "dir"])
    plt.suptitle(f"{site} — {event.start} to {event.stop}")
    plt.show()

## 3 · What the wind correction sees

At each buoy: Hs of every source; the wind sea the correction scales
(`Hs_ws`, from the contributions whose offshore forcing is a wind sea) and the
wind; how developed the wind sea is (g·Hs_ws/U², 0.24 = fully developed) and
the wind–wave angle, against the training range of 05b; and `K`.

In [ ]:
ws = mm.wind_sea_diagnostics(
    event, mda_n, metamodel=mm_set, resources=res, offshore_name=offshore_name, wl="forcing"
)
ranges = wind.training_range()
for site in show_buoys:
    hs = {"Buoy": buoy["hs"].sel(sites=site)}
    hs.update({label: ds["hs"].sel(sites=site) for label, ds in estimates.items() if label in metamodel or label == val.DYNAMIC_REFERENCE_LABEL})
    fig = plot_wind_sea_event(ws.sel(sites=site), hs, ranges, title=f"{site} — wind correction")
    plt.show()

## 4 · Maps at the peak

The hour of the event with the highest median dynamic Hs over the `map_set`
sites: the dynamic run, the metamodel without / with the correction minus it,
and `K`. The map set is reconstructed for the event's months if not cached yet
(resources below).

In [ ]:
map_res = mm.reconstruction_resources(
    event, mda_n, metamodel=map_set, spectral=True, offshore_name=offshore_name,
    wl="forcing", save_spectra=False, wind_correction=True,
)

In [ ]:
maps = {
    label: mm.reconstruct(
        event, mda_n, metamodel=map_set, spectral=True, offshore_name=offshore_name,
        wl="forcing", save_spectra=False, wind_correction=on, resources=map_res,
    )
    for on, label in wind_labels.items()
}
map_dynamic = dynamic["hs"].sel(sites=maps[wind_labels[False]].sites.values)
peak = map_dynamic.median("sites").idxmax("time").values
fig = plot_event_maps(
    map_dynamic.sel(time=peak),
    {label: ds["hs"].sel(time=peak) for label, ds in maps.items()},
    k=maps[wind_labels[True]]["wind_k"].sel(time=peak),
    reference_label=val.DYNAMIC_REFERENCE_LABEL,
    title=f"Hs at the peak, {pd.Timestamp(peak):%d %b %Y %H:%M}",
)
plt.show()